# 🏦 Exercícios SQL — CVM Data Lake
### Versão do Aluno | Big Data for Finance — FAE
**Banco:** PostgreSQL (DBeaver local)

**Tabelas utilizadas:**
- `layer_02_silver` — Tabelas que estão na camada Silver

**CNPJs de referência rápida:**
| Empresa | CNPJ |
|---|---|
| Petrobras | `33.000.167/0001-01` |
| Ambev | `07.526.557/0001-00` |
| Vale | `33.592.510/0001-54` |

---

## ⚙️ Configuração da Conexão

In [1]:
#importando as bibliotecas
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv

load_dotenv(encoding="utf-8")


True

In [2]:
# 1. Carrega o arquivo .env (2 níveis acima se estiver em notebooks/01_bronze/)
load_dotenv('../../.env')

# 2. Lê as variáveis de ambiente
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# 3. Encoding de segurança: transforma 'senha@123' em 'senha%40123'
encoded_pass = quote_plus(DB_PASS)

# 4. Monta e cria a engine SQLAlchemy
conn_string = f"postgresql+psycopg2://{DB_USER}:{encoded_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_string)

---
## Questão 1 — Primeira olhada na tabela Silver 🟢 Fácil
**Contexto:** A primeira coisa que um analista faz ao chegar em um novo banco é explorar o que existe dentro da tabela. A camada Silver já tem os dados do Balanço Patrimonial limpos, normalizados e prontos para análise.

**Pergunta:** Mostre as **10 primeiras linhas** da tabela `n1_dfp_cia_aberta_bp`, exibindo apenas as colunas: `CNPJ_CIA`, `DENOM_CIA`, `DT_REFER`, `CD_CONTA`, `DS_CONTA`, `VL_CONTA_TRATADO` e `IS_LEAF`.

In [8]:
# Definindo a consulta SQL com as colunas pedidas na Questão 1
query_q1 = """
SELECT 
    "CNPJ_CIA", 
    "DENOM_CIA", 
    "DT_REFER", 
    "CD_CONTA", 
    "DS_CONTA", 
    "VL_CONTA_TRATADO", 
    "IS_LEAF"
FROM layer_02_silver.n1_dfp_cia_aberta_bp
LIMIT 10;
"""

# Executando a consulta e salvando o resultado em um DataFrame do Pandas
with engine.connect() as conn:
    df_q1 = pd.read_sql(text(query_q1), conn)

# Exibindo as 10 primeiras linhas na tela do notebook
df_q1

,CNPJ_CIA,DENOM_CIA,DT_REFER,CD_CONTA,DS_CONTA,VL_CONTA_TRATADO,IS_LEAF
0,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1,ATIVO TOTAL,2.119934e+09,False
1,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01,ATIVO CIRCULANTE,4.499600e+08,False
2,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.01,CAIXA E EQUIVALENTES DE CAIXA,9.925800e+07,False
3,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.03,CONTAS A RECEBER,3.211700e+08,False
4,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.03.01,CLIENTES,3.211700e+08,False
5,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.03.01.01,CLIENTES,3.059410e+08,True
6,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.03.01.09,(-) PROVISÃO P/CRÉDITO DE LIQUIDAÇÃO DUVIDOSA,1.522900e+07,True
7,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.04,ESTOQUES,8.619000e+06,False
8,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.06,TRIBUTOS A RECUPERAR,2.091300e+07,False
9,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2010-12-31,1.01.06.01,TRIBUTOS CORRENTES A RECUPERAR,2.091300e+07,False


---
## Questão 2 — Volume e cobertura da base 🟢 Fácil
**Contexto:** Antes de qualquer análise, o analista precisa entender o tamanho e a cobertura da base: quantas linhas existem, quantas empresas distintas estão cobertas e qual o intervalo de datas disponível.

**Pergunta:** Em uma única query, calcule para a tabela `n1_dfp_cia_aberta_bp`:
- Total de linhas → `total_linhas`
- Quantidade de empresas distintas (por `CNPJ_CIA`) → `empresas_distintas`
- Quantidade de datas de referência distintas (`DT_REFER`) → `datas_distintas`

In [9]:
# Definindo a query para contar volume e cobertura
query_q2 = """
SELECT 
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT "CNPJ_CIA") AS empresas_distintas,
    COUNT(DISTINCT "DT_REFER") AS datas_distintas
FROM layer_02_silver.n1_dfp_cia_aberta_bp;
"""

# Executando a consulta e salvando o resultado no DataFrame
with engine.connect() as conn:
    df_q2 = pd.read_sql(text(query_q2), conn)

# Exibindo o resultado
df_q2

,total_linhas,empresas_distintas,datas_distintas
0,180461,185,54


---
## Questão 3 — Distribuição de empresas por setor 🟢 Fácil
**Contexto:** A CVM classifica cada empresa listada em um setor de atividade (`SETOR_ATIV`). Entender essa distribuição é o ponto de partida para qualquer análise setorial comparativa. Use a tabela `n0_empresas_selecionadas`, que contém o cadastro completo das empresas do nosso universo.

**Pergunta:** Quantas empresas existem em cada `SETOR_ATIV`? Ordene do setor com **mais** empresas para o com **menos**.

In [10]:
# Definindo a query para agrupar e contar as empresas por setor
query_q3 = """
SELECT 
    "SETOR_ATIV", 
    COUNT(*) AS quantidade_empresas
FROM layer_02_silver.n0_empresas_selecionadas
GROUP BY "SETOR_ATIV"
ORDER BY quantidade_empresas DESC;
"""

# Executando a consulta e salvando o resultado no DataFrame
with engine.connect() as conn:
    df_q3 = pd.read_sql(text(query_q3), conn)

# Exibindo o resultado final
df_q3

,SETOR_ATIV,quantidade_empresas
0,Energia Elétrica,33
1,"Construção Civil, Mat. Constr. e Decoração",30
2,Comércio (Atacado e Varejo),24
3,Serviços Transporte e Logística,17
4,Metalurgia e Siderurgia,13
5,Têxtil e Vestuário,13
6,"Máquinas, Equipamentos, Veículos e Peças",13
7,Alimentos,9
8,"Saneamento, Serv. Água e Gás",8
9,Serviços Médicos,8


---
## Questão 4 — Petrobras: Ativo e Passivo mais recentes 🟢 Fácil
**Contexto:** No Balanço Patrimonial da CVM, `CD_CONTA = '1'` é o **Ativo Total** e `CD_CONTA = '2'` é o **Passivo + Patrimônio Líquido Total**. A equação fundamental do BP é: **Ativo = Passivo + PL** — qualquer diferença indica erro de dados.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`), mostre o valor (em bilhões de reais) das contas `'1'` e `'2'` para a **data de referência mais recente** disponível. Use uma subconsulta escalar com `MAX("DT_REFER")` para encontrar a data mais recente.

*Dica: `ROUND((valor / 1e9)::NUMERIC, 2)` converte para bilhões com 2 casas.*

In [11]:
# Definindo a query com a subconsulta e conversão para bilhões
query_q4 = """
SELECT 
    "CNPJ_CIA",
    "DT_REFER",
    "CD_CONTA",
    "DS_CONTA",
    ROUND(("VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2) AS valor_bilhoes
FROM layer_02_silver.n1_dfp_cia_aberta_bp
WHERE "CNPJ_CIA" = '33.000.167/0001-01'
  AND "CD_CONTA" IN ('1', '2')
  AND "DT_REFER" = (
      SELECT MAX("DT_REFER") 
      FROM layer_02_silver.n1_dfp_cia_aberta_bp 
      WHERE "CNPJ_CIA" = '33.000.167/0001-01'
  );
"""

# Executando a consulta e salvando o resultado no DataFrame
with engine.connect() as conn:
    df_q4 = pd.read_sql(text(query_q4), conn)

# Exibindo o resultado final
df_q4

,CNPJ_CIA,DT_REFER,CD_CONTA,DS_CONTA,valor_bilhoes
0,33.000.167/0001-01,2024-12-31,1,ATIVO TOTAL,1124.8
1,33.000.167/0001-01,2024-12-31,2,PASSIVO TOTAL,1124.8


---
## Questão 5 — IS_LEAF: folhas vs. agregadores 🟢 Fácil
**Contexto:** `IS_LEAF = TRUE` marca contas **analíticas** (sem filhos) — são as únicas que devem ser somadas em qualquer ferramenta de BI, pois evitam dupla contagem. `IS_LEAF = FALSE` são contas **sintéticas** (pais) que já incorporam os valores de todas as suas contas filhas. Somar tudo geraria dupla contagem.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`) em `DT_REFER = '2023-12-31'`, quantas contas são `IS_LEAF = TRUE` e quantas são `IS_LEAF = FALSE`? Use `GROUP BY "IS_LEAF"`.

In [12]:
# Definindo a query para contar as contas analíticas (TRUE) e sintéticas (FALSE)
query_q5 = """
SELECT 
    "IS_LEAF", 
    COUNT(*) AS quantidade_contas
FROM layer_02_silver.n1_dfp_cia_aberta_bp
WHERE "CNPJ_CIA" = '33.000.167/0001-01'
  AND "DT_REFER" = '2023-12-31'
GROUP BY "IS_LEAF";
"""

# Executando a consulta e salvando o resultado no DataFrame
with engine.connect() as conn:
    df_q5 = pd.read_sql(text(query_q5), conn)

# Exibindo o resultado final
df_q5

,IS_LEAF,quantidade_contas
0,False,51
1,True,19


---
## Questão 6 — Top 10 maiores empresas por Ativo Total em 2023 🟡 Médio
**Contexto:** Rankings de tamanho de balanço são análises clássicas de mercado de capitais. O Ativo Total (`CD_CONTA = '1'`) resume o tamanho total de uma empresa — tudo que ela possui e tem direito a receber.

**Pergunta:** Liste as **10 empresas com maior Ativo Total** em `DT_REFER = '2023-12-31'`. Exiba `DENOM_CIA`, `SETOR_ATIV` e o valor em **bilhões** (2 casas decimais). Ordene do maior para o menor.

*Dica: `ROUND((expr)::NUMERIC, 2)` — o cast para `NUMERIC` é obrigatório no PostgreSQL quando a coluna é `DOUBLE PRECISION`.*

In [13]:
# Query para o Top 10 maiores empresas por Ativo Total
query_q6 = """
SELECT 
    t1."DENOM_CIA", 
    t2."SETOR_ATIV", 
    ROUND((t1."VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2) AS ativo_total_bilhoes
FROM layer_02_silver.n1_dfp_cia_aberta_bp t1
JOIN layer_02_silver.n0_empresas_selecionadas t2 
    ON t1."CNPJ_CIA" = t2."CNPJ_CIA"
WHERE t1."CD_CONTA" = '1' 
  AND t1."DT_REFER" = '2023-12-31'
ORDER BY ativo_total_bilhoes DESC
LIMIT 10;
"""

# Executando a consulta
with engine.connect() as conn:
    df_q6 = pd.read_sql(text(query_q6), conn)

# Exibindo o ranking
df_q6

,DENOM_CIA,SETOR_ATIV,ativo_total_bilhoes
0,PETROLEO BRASILEIRO S.A. PETROBRAS,Petróleo e Gás,1050.89
1,VALE S.A.,Extração Mineral,455.98
2,JBS S.A.,Alimentos,206.13
3,SUZANO S.A.,Papel e Celulose,143.59
4,COSAN S.A.,"Agricultura (Açúcar, Álcool e Cana)",139.87
5,AMBEV S.A.,Bebidas e Fumo,132.64
6,MARFRIG GLOBAL FOODS S.A.,Alimentos,130.95
7,TELEFÔNICA BRASIL S.A,Telecomunicações,120.74
8,EQUATORIAL S.A.,Energia Elétrica,103.64
9,BRASKEM S.A.,Petroquímicos e Borracha,91.74


---
## Questão 7 — Evolução anual do Ativo Total da Ambev 🟡 Médio
**Contexto:** Acompanhar a evolução do balanço ao longo dos anos revela a trajetória de crescimento (ou retração) de uma empresa. `DT_REFER` é do tipo `TIMESTAMP` — use `EXTRACT(YEAR FROM ...)` para extrair o ano.

**Pergunta:** Para a **Ambev** (`CNPJ_CIA = '07.526.557/0001-00'`), mostre a evolução anual do **Ativo Total** (`CD_CONTA = '1'`). Extraia o ano de `DT_REFER`, exiba o valor em bilhões e ordene do ano mais antigo ao mais recente.

In [14]:
# Query para a evolução anual do Ativo Total da Ambev
query_q7 = """
SELECT 
    EXTRACT(YEAR FROM "DT_REFER") AS ano,
    ROUND(("VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2) AS ativo_total_bilhoes
FROM layer_02_silver.n1_dfp_cia_aberta_bp
WHERE "CNPJ_CIA" = '07.526.557/0001-00'
  AND "CD_CONTA" = '1'
ORDER BY ano ASC;
"""

# Executando a consulta
with engine.connect() as conn:
    df_q7 = pd.read_sql(text(query_q7), conn)

# Exibindo o resultado
df_q7

,ano,ativo_total_bilhoes
0,2013.0,68.67
1,2014.0,72.14
2,2015.0,90.18
3,2016.0,83.84
4,2017.0,86.85
5,2018.0,94.13
6,2019.0,101.74
7,2020.0,125.20
8,2021.0,138.60
9,2022.0,137.96


---
## Questão 8 — Contas reconstruídas pelo pipeline 🟡 Médio
**Contexto:** `FLAG_RECONSTRUCAO = TRUE` indica que uma linha foi criada **sinteticamente** pelo pipeline — a empresa não reportou aquela conta e o pipeline reconstruiu o valor somando os filhos. Mais contas reconstruídas = mais "buracos" no dado original enviado à CVM.

**Pergunta:** Em **toda a base histórica**, quais são as **10 combinações de empresa e data** com mais contas reconstruídas no BP? Mostre `DT_REFER`, `DENOM_CIA` e o total de contas de cada empresa. Use `SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END)` para contar. Agrupe por `"DT_REFER"` e `"DENOM_CIA"`. Ordene da mais para a menos reconstruída.


In [15]:
# Query para identificar as empresas e datas com mais reconstruções sintéticas
query_q8 = """
SELECT 
    "DT_REFER", 
    "DENOM_CIA", 
    SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END) AS total_reconstruidas
FROM layer_02_silver.n1_dfp_cia_aberta_bp
GROUP BY "DT_REFER", "DENOM_CIA"
ORDER BY total_reconstruidas DESC
LIMIT 10;
"""

# Executando a consulta
with engine.connect() as conn:
    df_q8 = pd.read_sql(text(query_q8), conn)

# Exibindo o Top 10 de "buracos" no dado original
df_q8

,DT_REFER,DENOM_CIA,total_reconstruidas
0,2022-12-31,CESP - CIA ENERGETICA DE SAO PAULO,15
1,2021-12-31,VIVER INCORPORADORA E CONSTRUTORA S.A.,2
2,2017-12-31,CIA SIDERURGICA NACIONAL,1
3,2010-12-31,USINAS SID DE MINAS GERAIS S.A.-USIMINAS,1
4,2016-12-31,CIA SIDERURGICA NACIONAL,1
5,2021-12-31,INTERNATIONAL MEAL COMPANY ALIMENTACAO S.A.,1
6,2022-12-31,GRUPO CASAS BAHIA S.A.,1
7,2022-12-31,CIA BRASILEIRA DE DISTRIBUICAO,1
8,2021-12-31,CVC BRASIL OPERADORA E AGÊNCIA DE VIAGENS S.A.,1
9,2018-12-31,PRIO S.A.,0


---
## Questão 9 — Base analítica de Receita Bruta por empresa via JOIN 🟡 Médio
**Contexto:** Antes de agregar, o analista precisa da base analítica: uma linha por empresa com seu setor e receita. Com esse DataFrame em mãos, o Pandas consegue gerar estatísticas descritivas por grupo com `.describe()`.

**Pergunta:** Faça um `JOIN` entre `n1_dfp_cia_aberta_dre` e `n0_empresas_selecionadas`. Para `DT_REFER = '2023-12-31'` e `CD_CONTA = '3.01'` (Receita Bruta), traga uma linha por empresa com `CNPJ_CIA`, `DENOM_CIA`, `SETOR_ATIV` e o valor em bilhões. Depois use `.groupby` + `.describe()` para ver as estatísticas por setor.


In [16]:
# 1. Query SQL para criar a base analítica via JOIN
query_q9 = """
SELECT 
    t1."CNPJ_CIA", 
    t1."DENOM_CIA", 
    t2."SETOR_ATIV", 
    ROUND((t1."VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2) AS receita_bilhoes
FROM layer_02_silver.n1_dfp_cia_aberta_dre t1
JOIN layer_02_silver.n0_empresas_selecionadas t2 
    ON t1."CNPJ_CIA" = t2."CNPJ_CIA"
WHERE t1."DT_REFER" = '2023-12-31' 
  AND t1."CD_CONTA" = '3.01';
"""

# 2. Executando a consulta e trazendo para o Pandas
with engine.connect() as conn:
    df_q9 = pd.read_sql(text(query_q9), conn)

# 3. Usando o Pandas para ver as estatísticas por setor
estatisticas_setor = df_q9.groupby("SETOR_ATIV")["receita_bilhoes"].describe()

# Exibindo o resultado das estatísticas
estatisticas_setor

,count,mean,std,min,25%,50%,75%,max
SETOR_ATIV,,,,,,,,
"Agricultura (Açúcar, Álcool e Cana)",3.0,16.256667,20.594296,0.18,4.6500,9.120,24.2950,39.47
Alimentos,7.0,77.231429,134.756058,0.78,3.0350,10.840,79.5550,363.82
Bebidas e Fumo,1.0,79.740000,NaN,79.74,79.7400,79.740,79.7400,79.74
Bolsas de Valores/Mercadorias e Futuros,1.0,9.920000,NaN,9.92,9.9200,9.920,9.9200,9.92
Brinquedos e Lazer,3.0,1.700000,2.216213,0.16,0.4300,0.700,2.4700,4.24
Comunicação e Informática,7.0,2.040000,1.612245,0.33,0.9750,1.310,3.0950,4.50
Comércio (Atacado e Varejo),22.0,15.213636,34.877844,0.16,0.9650,2.785,12.4025,162.95
"Construção Civil, Mat. Constr. e Decoração",30.0,1.679000,2.049228,0.00,0.2650,1.035,2.3175,7.43
Educação,3.0,2.620000,0.989596,1.83,2.0650,2.300,3.0150,3.73
